# Scout Spark Delta Lake Quickstart


In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

In [ ]:
# Start spark session
spark = SparkSession.builder.appName("hl7-searches").enableHiveSupport().getOrCreate()

# --- Alternative: use this instead if your notebook server shuts down unexpectedly ---
# Large operations (joins, sorts, exports) write shuffle/temp data to the node's
# ephemeral storage. If this exceeds the pod's limit, Kubernetes evicts your pod —
# your notebook server will stop and you'll see a "Server not running" page in
# JupyterHub. To avoid this, redirect Spark temp data to your home directory, which
# uses your larger persistent storage volume.
# Comment out the session above, uncomment below, and restart your kernel.
#
# import os
# spark_tmp = os.path.expanduser("~/spark-tmp")
# os.makedirs(spark_tmp, exist_ok=True)
# spark = (
#     SparkSession.builder.appName("hl7-searches")
#     .config("spark.local.dir", spark_tmp)
#     .config(
#         "spark.driver.extraJavaOptions",
#         f"-Djava.io.tmpdir={spark_tmp} -Divy.cache.dir={spark_tmp} -Divy.home={spark_tmp}",
#     )
#     .enableHiveSupport()
#     .getOrCreate()
# )

In [ ]:
# Count the reports in the delta lake
# `reports` contains all report versions; `reports_latest` is deduplicated to the most recent.
# Note: some studies are read in parts and may be split into distinct reports;
# `reports_latest` will not capture those reports.
count_all = spark.sql("SELECT COUNT(*) as count_all FROM reports")
count_latest = spark.sql("SELECT COUNT(*) as count_latest FROM reports_latest")
count_all.show()
count_latest.show()

In [ ]:
# Count the number of reports per sending facility
all_facilities = spark.sql("""
    SELECT sending_facility, COUNT(*) as count
    FROM reports_latest
    GROUP BY sending_facility
    ORDER BY sending_facility ASC
""")
all_facilities.show(50, truncate=False)

# Count reports from IRB-approved institutions, adjust the list of facilities to match your IRB-approved institutions
FACILITIES = ['BJH', 'WUSM', 'BJCMG', 'SLCH']
FACILITIES_SQL = "'" + "', '".join(FACILITIES) + "'"

default_facility_count = spark.sql(f"""
    SELECT sending_facility, COUNT(*) as count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY sending_facility
    ORDER BY count DESC
""")
default_facility_count.show()

In [ ]:
# Show available modalities and count, filtering to selected facilities
modality_count = spark.sql(f"""
    SELECT modality, COUNT(*) as count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY modality
    ORDER BY count DESC
""")
modality_count.show()

In [ ]:
# Print out the schema and show an example report
# SELECT * here to show all report fields, but select only the fields you need for your analysis in practice
show_schema = spark.sql(f"""
    SELECT * FROM reports_latest WHERE sending_facility IN ({FACILITIES_SQL}) LIMIT 1
""")
show_schema.printSchema()
show_schema.show(1, vertical=True)

In [ ]:
# View sex distribution in CT and MR reports
sex_distribution_query = spark.sql(f"""
    SELECT sex, modality, COUNT(*) as count
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality IN ('CT', 'MR')
    GROUP BY sex, modality
    ORDER BY modality, sex, count DESC
""")
sex_distribution_query.show()

In [ ]:
# Filter reports by date range
# Relative: last 365 days
date_filtered = spark.sql(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND requested_dt >= current_date() - INTERVAL 365 DAYS
    LIMIT 10
""")
date_filtered.show(truncate=40)

# Absolute: specific date range
date_filtered_abs = spark.sql(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND requested_dt BETWEEN DATE '2024-01-01' AND DATE '2026-01-01'
    LIMIT 10
""")
date_filtered_abs.show(truncate=40)

In [ ]:
# Search report text using regex
text_search = spark.sql(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND REGEXP_LIKE(report_text, '(?i)pulmonary embolism')
    LIMIT 10
""")
text_search.show(truncate=40)

In [ ]:
# Filter by diagnosis code
dx_search = spark.sql(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, diagnoses
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND exists(diagnoses, e -> e.diagnosis_code IN ('I26.99', 'I26.9'))
    LIMIT 10
""")
dx_search.show(truncate=40)

In [ ]:
# Combine multiple filters: CT head scans for adults in 2024
combined_search = spark.sql(f"""
    SELECT accession_number, epic_mrn, patient_age, sex, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality = 'CT'
      AND REGEXP_LIKE(service_name, '(?i)head|brain')
      AND patient_age >= 18
      AND requested_dt >= DATE '2024-01-01'
    LIMIT 10
""")
combined_search.show(truncate=40)

In [ ]:
# Show all available patient ID types from the patient_ids array
# Each report has a patient_ids array of structs with: id_number, assigning_authority, identifier_type_code, assigning_facility
# The transformer also creates convenience columns like epic_mrn, empi_mr, bjh_mr, slch_mr, etc.
patient_id_types = spark.sql(f"""
    SELECT pid.assigning_authority, pid.identifier_type_code, COUNT(*) as count
    FROM reports_latest
    LATERAL VIEW explode(patient_ids) AS pid
    WHERE sending_facility IN ({FACILITIES_SQL})
    GROUP BY pid.assigning_authority, pid.identifier_type_code
    ORDER BY count DESC
""")
patient_id_types.show(20, truncate=False)

In [ ]:
# Random sample of CTs
sample = spark.sql(f"""
    SELECT epic_mrn, birth_date, sex, race, modality, service_name,
           requested_dt, study_instance_uid, report_text
    FROM reports_latest 
    WHERE sending_facility IN ({FACILITIES_SQL})
      AND modality='CT'
    ORDER BY RAND() 
    LIMIT 100
""")
print(f"Sampled {sample.count()} reports")

In [ ]:
# Convert Spark DataFrame to pandas for familiar data manipulation
df = sample.toPandas()
df.head()

In [ ]:
# Import a list of patient IDs from a CSV file to look up their reports
# Upload your own CSV to /home/jovyan/Scout/ using the JupyterHub file browser
# CSV should have a column named 'epic_mrn' — you can also join on
# study_instance_uid, empi_mr, or any other ID column in the reports_latest table
import os

patient_ids_path = '/home/jovyan/Scout/patient_ids.csv'
if not os.path.exists(patient_ids_path):
    pd.DataFrame({'epic_mrn': ['REPLACE_WITH_REAL_MRN']}).to_csv(patient_ids_path, index=False)
    print(f"Created sample CSV at {patient_ids_path} — edit it with your own patient IDs")

patient_ids_df = pd.read_csv(patient_ids_path, dtype=str)
mrns = patient_ids_df['epic_mrn'].dropna().str.strip().tolist()
print(f"Loaded {len(mrns)} patient IDs from CSV")

In [ ]:
# Look up reports for the patient list
mrn_list = "', '".join(mrns)

search = spark.sql(f"""
    SELECT accession_number, epic_mrn, modality, service_name, requested_dt, report_text
    FROM reports_latest
    WHERE epic_mrn IN ('{mrn_list}')
      AND sending_facility IN ({FACILITIES_SQL})
    LIMIT 100
""")
search.show(5, truncate=40)

In [ ]:
# Write results to csv
search.write.csv('/home/jovyan/Scout/patient_list_results_csv', header=True)

## Installing Additional Packages

The base Jupyter environment includes PySpark, Delta Lake, Trino, pandas, matplotlib, seaborn, and other core data analysis packages. For ML, NLP, or other specialized work, create a new conda environment:

```bash
# Create an environment with specific packages
mamba create -n my-env python=3.11 ipykernel pytorch transformers scikit-learn -y

# Or use the sample environment file (in ~/Scout/environment.yml)
mamba env create -f ~/Scout/environment.yml
```

**Important:** Include `ipykernel` in every environment you create. The `nb_conda_kernels` extension uses it to discover the environment as a Jupyter kernel. Without it, your environment won't appear in the kernel list.

After creating an environment, refresh the launcher to see it as a selectable kernel.

Environments are stored on your persistent home directory (`/home/jovyan/.conda/envs/`) and survive server restarts. In air-gapped deployments, package requests are routed through a proxy transparently — no extra configuration needed.

For more details, see the [Tips & Best Practices](https://washu-scout.readthedocs.io/en/latest/tips.html#installing-additional-packages) documentation.